In [1]:
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import os


Define parameters


In [8]:
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io
import sys
import os


def convert_utc_to_local(df, local_tz):
    """
    Convert the datetime index of the DataFrame from UTC to a local timezone.

    Args:
    df : pandas.DataFrame
        DataFrame with a datetime index in UTC.
    local_tz : str
        A timezone string (e.g., 'America/Chicago').

    Returns:
    pandas.DataFrame
        DataFrame with datetime index converted to the specified local timezone.
    """
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df = df.reset_index(level='station', drop=True)
        df.index = pd.to_datetime(df.index)

    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    
    df.index = df.index.tz_convert(local_tz)
    return df

def filter_dataframe_by_date(df, start_date, end_date, timezone=None):
    """
    Filter the DataFrame to include rows between the specified start and end dates,
    handling timezone differences appropriately.
    """
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    if timezone:
        start_date = start_date.tz_localize(timezone)
        end_date = end_date.tz_localize(timezone)
    else:
        df.index = df.index.tz_localize(None)

    return df.loc[(df.index >= start_date) & (df.index <= end_date)]

def get_parameters_MERRA2(lat, lon, year):
    api_endpoint = f"https://power.larc.nasa.gov/api/temporal/hourly/point?community=SB&parameters=&longitude={lon}&latitude={lat}&start={year}0101&end={year}1231&format=EPW"
    response = requests.get(api_endpoint)
    csv_data = io.StringIO(response.text)
    df = pd.read_csv(csv_data, skiprows=8, header=None)
    header = '\n'.join(response.text.splitlines()[:8])

    # Check if the dataframe has more than 8761 rows and truncate if necessary
    # Sometimes MERRA2 erroneously provides extra rows
    if len(df) > 8760:
        df = df.iloc[:8760]


    return df, header

# def merge_data(df, data):
#     """
#     Merges two datasets, replacing specific columns in `df` with corresponding values from `data`.
#     """

#     print(df.head(5))
#     for col in [6, 7, 8, 33, 30, 21, 20, 9]:  # Replace indices with more descriptive names if possible
#         if not df[col].isna().all():
#             df[col] = list(data['temp'][1:])
#     print(df.head(5))
#     return df


def merge_data(df, data):
    # Tdb
    if not data['temp'].isna().all():
        df[6] = list(data['temp'][1:])
    # Tdew
    if not data['dwpt'].isna().all():
        df[7] = list(data['dwpt'][1:])
    # RH
    if not data['rhum'].isna().all():
        df[8] = list(data['rhum'][1:])
    # Precep
    if not data['prcp'].isna().all():
        df[33] = list(data['prcp'][1:])
    # Snow
    if not data['snow'].isna().all():
        df[30] = list(data['snow'][1:])
    # Wspeed
    if not data['wspd'].isna().all():
        df[21] = list(data['wspd'][1:])
    # Wdir
    if not data['wdir'].isna().all():
        df[20] = list(data['wdir'][1:])
    # P, go from hPa to Pa
    if not data['pres'].isna().all():
        df[9] = [x * 100 for x in list(data['pres'][1:])]

    return df

def check_missing_hours(year, df):
    """
    Checks for missing hours in the DataFrame's datetime index for a specified year.
    """
    full_index = pd.date_range(start=f"{year}-01-01", end=f"{year+1}-01-01", freq="H")
    missing_hours = full_index.difference(df.index)
    missing_hours_num = len(missing_hours)

    if missing_hours_num > 0:
        diffs = missing_hours.to_series().diff().dt.total_seconds().div(3600)
        largest_consecutive_group = (diffs != 1).cumsum().value_counts().max()
    else:
        largest_consecutive_group = 0

    return missing_hours_num, largest_consecutive_group

def fix_wmo(wmo):
    """
    Attempts to fix or standardize the WMO code format.
    """
    if wmo:
        try:
            return str(int(wmo))
        except ValueError:
            icao = wmo
            return get_wmo_from_icao_NOAA(icao) or icao
    return wmo

def get_noaa_merra2_data(lat, lon, year, file_type, save_folder):
    """
    Retrieves NOAA and MERRA2 data for a specific location and year.
    """
    retrieve_status = True
    data_noaa, tz, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    if epw_exists:
        df_merged = ''
        retrieve_status = False
        distance = ''
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = ''
        epw_exists = True
        return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    elif incomplete_timeseries:
        df_merged = ''
        retrieve_status = False
        distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        wmo = ''
        epw_exists = False
        return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    try:
        data_noaa_tz_adj = filter_dataframe_by_date(convert_utc_to_local(data_noaa, tz), datetime(year, 1, 1), datetime(year+1, 1, 1))
    except AttributeError:
        # print("We don't have NOAA data for this location/year")
        df_merged = ''
        retrieve_status = False
        distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        epw_exists = False
        return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    info_dict = {
    'timeshift': get_time_shift(tz),
    'elevation': elevation,
    'wmo': wmo,
    'station_name': station_name,
    'state': state,
    'country': country,
    'lat': latitude_station,
    'lon': longitude_station,
    'weather_file_type': file_type
    }

    data_noaa_tz_adj_h = data_noaa_tz_adj.resample('H').mean()
    data_noaa_tz_adj_h_interpolated = data_noaa_tz_adj_h.interpolate(method='linear', limit=3, limit_direction='forward')
    hdd, cdd = calculate_hdd_cdd(data_noaa_tz_adj_h_interpolated, 'temp')
    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)

    df_merged = merge_data(df_merra2, data_noaa_tz_adj_h_interpolated)

    return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

def run_individual_location(lat, lon, year, file_type, save_folder, save_name):
    """
    Processes a single location, fetching data and handling errors.
    """
    data_meteostat_merra2, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists = get_noaa_merra2_data(lat, lon, year, file_type, save_folder)
    if epw_exists:
        retrieve_status = False
        distance = ''
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        retrieve_info_closest_other_locations = True
        # return retrieve_status, distance, info_dict['wmo'], hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations
    elif retrieve_status:
        retrieve_info_closest_other_locations = False
        #Save the EPW file
        if save_name != None:
            output_path = os.path.join(save_folder, f"{save_name.replace(' ', '_').replace('.', '_')}_{year}.epw")
        else:
            output_path = os.path.join(save_folder, f"{wmo}_{year}.epw")
        data_meteostat_merra2.to_csv(output_path, header=False, index=False)
        with open(output_path, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_path, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
    else:
        retrieve_info_closest_other_locations = False
        retrieve_status = False
        print('No data available for this location/year.')

    return retrieve_status, distance, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations

def get_time_shift(timezone_name):
    """
    Calculates the time shift for a given timezone from UTC.
    """
    timezone = pytz.timezone(timezone_name)
    now = datetime.now(timezone)
    utc_offset = now.utcoffset()
    return int(utc_offset.total_seconds() // 3600)  # Return hours offset only

def calculate_hdd_cdd(df, temperature_column):
    """
    Calculate Heating Degree Days (HDD) and Cooling Degree Days (CDD) from hourly temperature data in Celsius.
    """
    df[temperature_column + '_F'] = df[temperature_column] * 9 / 5 + 32
    base_temperature = 65

    df['date'] = df.index.to_series().dt.date
    daily_mean_temp = df.groupby('date')[temperature_column + '_F'].mean().reset_index()
    daily_mean_temp.columns = ['date', 'mean_temp']

    daily_mean_temp['HDD'] = (base_temperature - daily_mean_temp['mean_temp']).clip(lower=0)
    daily_mean_temp['CDD'] = (daily_mean_temp['mean_temp'] - base_temperature).clip(lower=0)

    total_hdd = daily_mean_temp['HDD'].sum()
    total_cdd = daily_mean_temp['CDD'].sum()

    return int(total_hdd), int(total_cdd)

def check_epw_exists(save_folder, year, wmo):
    return os.path.exists(f'{save_folder}/{wmo}_{year}.epw')

def create_header(df, year, info_dict):
    header_lines = []

    #Calculated parameters 
    first_day_year = pd.to_datetime(date.min.replace(year=year)).day_name()
    leap_status = lambda year: 'Yes' if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 'No'
    dst_start, dst_end = get_dst_start_end(year, info_dict['lat'], info_dict['lon'])
    design_conditions_file = 'resources/design_conditions.csv'
    design_conditions_line = find_closest_design_condition(float(info_dict['lat']), float(info_dict['lon']), design_conditions_file)
    #Hardcoded parameters
    number_of_holidays = 0
    number_of_data_periods = 1
    number_of_records_per_hour = 1

    # line_1
    header_lines.append(f"LOCATION,{info_dict['station_name']},{info_dict['state']},{info_dict['country']},{info_dict['weather_file_type']},{info_dict['wmo']},{info_dict['lat']},{info_dict['lon']},{info_dict['timeshift']},{info_dict['elevation']}")
    # line_2
    header_lines.append(design_conditions_line)
    # line_3
    header_lines.append(f"TYPICAL/EXTREME PERIODS,0")
    # line_4
    header_lines.append(f"GROUND TEMPERATURES,0")
    # header_lines.append(f"GROUND TEMPERATURES,3,.5,,,,-16.34,-17.80,-15.22,-11.16,-0.57,7.61,13.13,14.81,11.95,5.60,-2.89,-10.76,2,,,,-10.97,-13.57,-13.04,-10.89,-3.80,2.61,7.74,10.49,9.90,6.30,0.46,-5.74,4,,,,-6.53,-9.19,-9.78,-8.97,-4.96,-0.64,3.32,6.08,6.72,5.16,1.73,-2.4")
    # line_5
    try:
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},{dst_start.month}/{dst_start.day},{dst_end.month}/{dst_end.day},{number_of_holidays}")
    except AttributeError:
        #We cannot retrieve DST dates, let's set them to 0
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},0,0,{number_of_holidays}")
    # line_6
    header_lines.append(f"COMMENTS 1, ")
    # line_7
    header_lines.append(f"COMMENTS 2, ")
    # line_8
    header_lines.append(f"DATA PERIODS,{number_of_data_periods},{number_of_records_per_hour},Data,{first_day_year},{df.iloc[0, 1]}/{df.iloc[0, 2]},{df.iloc[-1, 1]}/{df.iloc[-1, 2]}")

    return header_lines

def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)

    epw_exists = False
    station_number = 0
    len_data = 0

    incomplete_timeseries = True
    while incomplete_timeseries:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[-1]))
        # First check if EPW already exists
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            incomplete_timeseries = False
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data)
        if (len_data > 8000) & (largest_consecutive_group <= 3):
            incomplete_timeseries = False
        distance = stations.fetch(station_number)['distance'].values[-1]
        # Let's stop after 100mi
        if distance > 160000:
            break

    if epw_exists | incomplete_timeseries:
        data = ''
        timezone = ''
        distance = ''
        elevation = ''
        station_name = ''
        state = ''
        country = ''
        latitude_station = ''
        longitude_station = ''
                
    else:
        station_info = stations.fetch(station_number)
        timezone = station_info['timezone'].values[-1]
        elevation = station_info['elevation'].values[-1]
        distance = stations.fetch()['distance'].values[-1]
        wmo = fix_wmo(str(station_info.index.values[-1]))
        station_name = station_info['name'].values[-1]
        state = station_info['region'].values[-1]
        country = station_info['country'].values[-1]
        latitude_station = station_info['latitude'].values[-1]
        longitude_station = station_info['longitude'].values[-1]

    return data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries

# Helper function to check and update missing data
def update_if_missing(df, index, col_name, new_value):
    if pd.isna(df.at[index, col_name]) or not df.at[index, col_name]:
        df.at[index, col_name] = new_value

def retrieve_info_other_location(wmo, zipcodes, year):
    retrieve_status = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"Do we have data for {year}?"].values[0]
    distance = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"distance_location_station_miles_{year}"].values[0]
    hdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"hdd_base65F_{year}"].values[0]
    cdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"cdd_base65F_{year}"].values[0]
    return retrieve_status, distance, hdd, cdd

def get_wmo_from_icao_NOAA(icao_code):
    # URL to the NOAA ISD database metadata
    url = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"
    
    # Fetch the data
    response = requests.get(url)
    
    if response.status_code == 200:
        # Read the CSV data
        csv_data = response.content.decode('utf-8')
        
        # Parse the CSV data
        lines = csv_data.splitlines()
        headers = lines[0].split(',')
        icao_index = headers.index('"ICAO"')
        wmo_index = headers.index('"USAF"')

        for line in lines[1:]:
            fields = line.split(',')
            if fields[icao_index].strip('"') == icao_code.upper():
                return fields[wmo_index].strip('"')
    else:
        print(f"Failed to retrieve data, status code: {response.status_code}")
        return None

def get_dst_start_end(year, latitude, longitude):
    # Get the timezone for the given latitude and longitude
    tf = TimezoneFinder()
    timezone_str = tf.timezone_at(lat=latitude, lng=longitude)
    
    if timezone_str is None:
        raise ValueError("Could not find timezone for the given coordinates.")
    
    # Get the timezone object
    timezone = pytz.timezone(timezone_str)
    
    # Define the dates for the beginning and end of the year (naive datetime)
    start_of_year = datetime(year, 1, 1)
    end_of_year = datetime(year, 12, 31)
    
    dst_start = None
    dst_end = None

    # Start by localizing the first date
    previous_offset = timezone.localize(start_of_year).dst()

    # Loop through each day of the year
    for dt in [start_of_year + timedelta(days=i) for i in range((end_of_year - start_of_year).days + 1)]:
        localized_dt = timezone.localize(dt)  # Localize naive datetime
        current_offset = localized_dt.dst()
        
        if previous_offset == timedelta(0) and current_offset != timedelta(0):
            dst_start = localized_dt
        elif previous_offset != timedelta(0) and current_offset == timedelta(0):
            dst_end = localized_dt
            break
        
        previous_offset = current_offset
    
    return dst_start, dst_end

def find_closest_design_condition(lat,lon,design_conditions_file):
    """
    Finds the closest design condition from the CSV file based on the given latitude and longitude.

    Parameters:
    lat (float): The latitude of the target location.
    lon (float): The longitude of the target location.
    csv_file (str): The path to the CSV file containing design conditions.

    Returns:
    str: The 2021 design condition string for the closest location.
    """

    # Function to calculate the distance between two points given their latitudes and longitudes
    def haversine_distance(lat1, lon1, lat2, lon2):
        # Convert latitude and longitude from degrees to radians
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        
        # Haversine formula
        dlat = lat2 - lat1 
        dlon = lon2 - lon1 
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a)) 
        r = 6371  # Radius of Earth in kilometers. Use 3956 for miles. Determines return value units.
        return c * r

    
    # Read the CSV file
    df = pd.read_csv(design_conditions_file)

    # Calculate distance from target coordinates to each row in the dataframe
    df['distance'] = df.apply(lambda row: haversine_distance(lat, lon, row['latitude'], row['longitude']), axis=1)

    # Find the row with the minimum distance
    closest_row = df.loc[df['distance'].idxmin()]

    # Return the design conditions for 2021
    return closest_row['2021_design_conditions']



# Define constants
year = 2022
file_type = 'AMY'
save_folder = 'epws_wmo'

# Check if the 'zipcodes' variable is already defined
if 'zipcodes' not in globals():
    # Load the zip codes CSV only if 'zipcodes' is not already defined
    zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# Initialize a counter for iterations
counter = 0

# Process each row in the DataFrame starting from the specified index
for index, row in zipcodes.iloc[0:].iterrows():
    print(index)

    # if row.get(f"Do we have data for {year}?") == 'True':
    #     continue

    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed
    lat = row['lat']
    lon = row['lng']
    save_name = None

    # print(lat)
    # print(lon)
    # print(row['city'])

    # Retrieve data for the current location
    retrieve_status, distance, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    if retrieve_info_closest_other_locations:
        try:
            retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
        except IndexError:
            print(wmo)
            retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)


    # Update the DataFrame only if the cell is empty or contains a placeholder (like 'nan')
    update_if_missing(zipcodes, index, f"Do we have data for {year}?", retrieve_status)
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance * 0.000621371)  # Convert from meters to miles
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)

    # Increment the counter
    counter += 1

    # Every 10 iterations, save the DataFrame and reopen it
    if counter % 10 == 0:
        # Save the DataFrame to the CSV file
        zipcodes.to_csv('resources/zip_code_list.csv', index=False)

        # Reopen the file to ensure the latest version is loaded
        # zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# After the loop is done, ensure the latest state is saved
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

# KVHN0
# KSUN
# 74611


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114


115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140


141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
281
282
283
284
285
286
287
288
289
290
291
292
293
294
295
296
297
298
299
300
301
302
303
304
305
306
307


308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325
326
327
328
329
330
331
332
333
334
335
336
337
338
339
340
341
342
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367
368
369
370
371
372
373
374
375
376
377
378
379
380
381
382
383
384
385
386
387
388
389
390
391
392
393
394
395
396
397
398
399
400
401
402
403
404
405
406
407
408
409
410
411
412
413
414
415
416
417
418
419
420
421
422
423
424
425
426
427
428
429
430
431
432
433
434
435
436
437
438
439
440
441
442
443
444
445
446
447
448
449
450
451
452
453
454
455
456
457
458
459
460
461
462
463
464
465
466
467
468
469
470
471
472
473
474
475
476
477
478
479
480
481
482
483
484
485
486
487
488
489
490
491
492
493
494
495
496
497
498
499
500
501
502
503
504
505
506
507
508
509
510
511
512
513
514
515
516
517
518
519
520
521
522
523
524
525
526
527
528
529
530
531
532
533
534
535
536
537
538
539
540
541
542
543
544
545
546
547
548
549
550
551
552
553
554
555
556
557


KeyboardInterrupt: 

In [53]:
meteostat_df = pd.read_csv('resources/meteostat_stats.csv')
zip_row = pd.read_csv('resources/zip_code_list.csv')
zip_row


,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,...,military,timezone,Distance [km],Distamce [mi],Location,zip0,Do we have data for 2022?,weather_station_wmo_2022,hdd_base65F_2022,cdd_base65F_2022
0,59444,48.72526,-111.36528,Galata,MT,Montana,True,NaN,223,0.3,...,False,America/Denver,75.359917,46.807402,Cut Bank / Little Browning,59444,True,ZEZ8W,8643.0,410.0
1,59466,48.75838,-111.66670,Oilmont,MT,Montana,True,NaN,133,0.3,...,False,America/Denver,54.684008,33.965222,Cut Bank / Little Browning,59466,True,71345,8536.0,358.0
2,59482,48.87288,-111.90786,Sunburst,MT,Montana,True,NaN,589,0.7,...,False,America/Denver,45.208574,28.079860,Cut Bank / Little Browning,59482,True,71244,8877.0,281.0
3,59544,48.84308,-107.55539,Whitewater,MT,Montana,True,NaN,231,0.2,...,False,America/Denver,59.325437,36.848097,Malta / Riverside Trailer Court,59544,True,71137,9736.0,443.0
4,59250,48.85642,-106.57354,Opheim,MT,Montana,True,NaN,172,0.1,...,False,America/Denver,71.204559,44.226434,Glasgow International Airport,59250,True,ZRBBD,9801.0,369.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32961,99627,63.53161,-154.62942,McGrath,AK,Alaska,True,NaN,308,0.0,...,False,America/Anchorage,37.164580,23.083590,Farewell / Intermediate Field,99627,False,NaN,NaN,NaN
32962,99638,52.89995,-168.93738,Nikolski,AK,Alaska,True,NaN,26,0.5,...,False,America/Nome,8.079798,5.018508,Nikolski,99638,False,NaN,NaN,NaN
32963,99691,62.86380,-153.66044,Nikolai,AK,Alaska,True,NaN,114,0.0,...,False,America/Anchorage,35.779275,22.223152,Farewell Lake / Nikolai,99691,False,NaN,NaN,NaN
32964,99701,67.16977,-149.56917,Fairbanks,AK,Alaska,True,NaN,17510,1.6,...,False,America/Anchorage,59.581889,37.007385,Chandalar Lake,99701,False,NaN,NaN,NaN


In [54]:
zip_row.columns

Index(['zip', 'lat', 'lng', 'city', 'state_id', 'state_name', 'zcta',
       'parent_zcta', 'population', 'density', 'county_fips', 'county_name',
       'county_weights', 'county_names_all', 'county_fips_all', 'imprecise',
       'military', 'timezone', 'Distance [km]', 'Distamce [mi]', 'Location',
       'zip0', 'Do we have data for 2022?', 'weather_station_wmo_2022',
       'hdd_base65F_2022', 'cdd_base65F_2022'],
      dtype='object')

In [52]:
import math

# Function to calculate the distance between two lat/lon points using the Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    # Radius of the Earth in miles
    R = 3958.8
    
    # Convert degrees to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    # Haversine formula
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    # Distance in miles
    return R * c


# Iterate over each row in zip_df
for idx, zip_row in zipcodes.iterrows():
    # print(idx)

    if bool(zip_row['Do we have data for 2022?']):
        wmo_code = zip_row['weather_station_wmo_2022']
        
        # Look for a matching row in meteostat_df using WMO or ICAO code
        # station_row = meteostat_df[(meteostat_df['wmo'].astype(str) == wmo_code) | (meteostat_df['icao'].astype(str) == wmo_code)]
        try:
            station_row = meteostat_df[(meteostat_df['wmo'] == int(wmo_code))]
        except ValueError:
            try:
                station_row = meteostat_df[(meteostat_df['icao'].astype(str) == wmo_code[:4])]
            except TypeError:
                print('----------------------------')
                print(wmo_code)
                print(zip_row['zip0'])
                print(zip_row['Do we have data for 2022?'])
                continue
                # station_row = meteostat_df[(meteostat_df['icao'].astype(str) == wmo_code[:4])]
        
        if not station_row.empty:
            # Extract lat/lon from meteostat_df
            lat_stat = station_row['latitude'].values[0]
            lon_stat = station_row['longitude'].values[0]
            
            # Extract lat/lon from zip_df
            lat_zip = zip_row['lat']
            lon_zip = zip_row['lng']
            
            # Calculate the distance in miles
            distance = haversine(lat_zip, lon_zip, lat_stat, lon_stat)
            
            # Save the calculated distance in the new column
            zipcodes.at[idx, 'distance_location_station_miles_2022'] = distance

# Show the updated zip_df with distances
zipcodes.head()


----------------------------
nan
59001
False
----------------------------
nan
69128
False


,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,...,Distance [km],Distamce [mi],Location,zip0,Do we have data for 2022?,distance_location_station_miles_2022,weather_station_wmo_2022,hdd_base65F_2022,cdd_base65F_2022,distance_2022_miles
0,1001,42.06259,-72.62589,Agawam,MA,Massachusetts,True,NaN,17621,591.4,...,12.907752,8.017237,Westfield / Owen District,1001,True,8.020595,KBAF0,5983.0,802.0,None
1,1002,42.37492,-72.46210,Amherst,MA,Massachusetts,True,NaN,30066,210.8,...,20.312900,12.616708,Chicopee Falls / Westover Air Force Base,1002,True,12.621992,74491,5980.0,740.0,None
2,1003,42.39192,-72.52479,Amherst,MA,Massachusetts,True,NaN,11238,6099.7,...,21.352006,13.262115,Chicopee Falls / Westover Air Force Base,1003,True,13.267669,74491,5980.0,740.0,None
3,1005,42.42017,-72.10615,Barre,MA,Massachusetts,True,NaN,4991,43.5,...,22.398644,13.912201,Orange / Partridgeville,1005,True,13.918027,KORE0,6399.0,648.0,None
4,1007,42.27875,-72.40036,Belchertown,MA,Massachusetts,True,NaN,14967,109.9,...,14.015979,8.705577,Chicopee Falls / Westover Air Force Base,1007,True,8.709223,74491,5980.0,740.0,None


In [42]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)


In [11]:
def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)

    epw_exists = False
    station_number = 0
    len_data = 0

    incomplete_timeseries = True
    while incomplete_timeseries:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[-1]))
        # First check if EPW already exists
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            incomplete_timeseries = False
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data)
        if (len_data > 8000) & (largest_consecutive_group <= 3):
            incomplete_timeseries = False
        distance = stations.fetch(station_number)['distance'].values[-1]
        # Let's stop after 100mi
        if distance > 160000:
            break

    if epw_exists | incomplete_timeseries:
        data = ''
        timezone = ''
        distance = ''
        elevation = ''
        station_name = ''
        state = ''
        country = ''
        latitude_station = ''
        longitude_station = ''
                
    else:
        station_info = stations.fetch(station_number)
        timezone = station_info['timezone'].values[-1]
        elevation = station_info['elevation'].values[-1]
        distance = stations.fetch()['distance'].values[-1]
        wmo = fix_wmo(str(station_info.index.values[-1]))
        station_name = station_info['name'].values[-1]
        state = station_info['region'].values[-1]
        country = station_info['country'].values[-1]
        latitude_station = station_info['latitude'].values[-1]
        longitude_station = station_info['longitude'].values[-1]

    return data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries


lat = 42.06259
lon = -72.62589

data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries = get_data_noaa(lat, lon, 2022, '')
data.head(5)

,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
time,,,,,,,,,,,
2021-12-31 00:00:00,5.6,3.9,89.0,0.0,NaN,0.0,0.0,NaN,1014.3,NaN,NaN
2021-12-31 01:00:00,5.0,4.4,96.0,0.3,NaN,0.0,0.0,NaN,1014.4,NaN,NaN
2021-12-31 02:00:00,5.0,4.4,96.0,0.5,NaN,0.0,0.0,NaN,1014.3,NaN,5.0
2021-12-31 03:00:00,4.4,4.0,97.0,0.3,NaN,0.0,0.0,NaN,1013.8,NaN,5.0
2021-12-31 04:00:00,4.4,4.0,97.0,0.3,NaN,0.0,0.0,NaN,1013.9,NaN,5.0


In [12]:
data.interpolate(method='linear', limit=3, limit_direction='forward')

,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
time,,,,,,,,,,,
2021-12-31 00:00:00,5.6,3.9,89.0,0.0,NaN,0.0,0.0,NaN,1014.3,NaN,NaN
2021-12-31 01:00:00,5.0,4.4,96.0,0.3,NaN,0.0,0.0,NaN,1014.4,NaN,NaN
2021-12-31 02:00:00,5.0,4.4,96.0,0.5,NaN,0.0,0.0,NaN,1014.3,NaN,5.0
2021-12-31 03:00:00,4.4,4.0,97.0,0.3,NaN,0.0,0.0,NaN,1013.8,NaN,5.0
2021-12-31 04:00:00,4.4,4.0,97.0,0.3,NaN,0.0,0.0,NaN,1013.9,NaN,5.0
...,...,...,...,...,...,...,...,...,...,...,...
2023-01-01 20:00:00,6.7,-0.5,60.0,0.0,NaN,290.0,18.4,NaN,1013.4,NaN,3.0
2023-01-01 21:00:00,5.6,-1.1,62.0,0.0,NaN,0.0,0.0,NaN,1014.1,NaN,3.0
2023-01-01 22:00:00,5.6,-1.1,62.0,0.0,NaN,0.0,0.0,NaN,1014.1,NaN,3.0


In [11]:
zipcodes.head(20)

,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,...,timezone,Distance [km],Distamce [mi],Location,zip0,Do we have data for 2022?,distance_location_station_miles_2022,weather_station_wmo_2022,hdd_base65F_2022,cdd_base65F_2022
0,96799,-14.21967,-170.36930,Pago Pago,AS,American Samoa,True,NaN,NaN,NaN,...,Pacific/Pago_Pago,4036.005334,2506.835611,South Kona / Hawaii,96799,NaN,NaN,NaN,NaN,NaN
1,99764,63.38147,-141.51133,Northway,AK,Alaska,True,NaN,439.0,0.1,...,America/Anchorage,132.362820,82.212932,Nabesna,99764,NaN,NaN,NaN,NaN,NaN
2,99923,55.97796,-130.03671,Hyder,AK,Alaska,True,NaN,15.0,0.4,...,America/Sitka,126.398396,78.508320,Ketchikan International Airport,99923,NaN,NaN,NaN,NaN,NaN
3,86510,36.23729,-110.23069,Pinon,AZ,Arizona,True,NaN,4693.0,2.4,...,America/Denver,123.467518,76.687900,Window Rock Airport,86510,NaN,NaN,NaN,NaN,NaN
4,89832,41.98653,-116.16574,Owyhee,NV,Nevada,True,NaN,1351.0,3.9,...,America/Los_Angeles,119.974313,74.518207,Mountain Home / Mountain Home Air Force Base,89832,NaN,NaN,NaN,NaN,NaN
5,86039,36.06276,-110.53209,Kykotsmovi Village,AZ,Arizona,True,NaN,1273.0,1.7,...,America/Denver,115.682681,71.852597,Winslow Municipal Airport,86039,NaN,NaN,NaN,NaN,NaN
6,704,17.96719,-66.22013,Aguirre,PR,Puerto Rico,True,NaN,8274.0,655.0,...,America/Puerto_Rico,113.185255,70.301401,"Aquadilla, Rafael Hernandez Airport",704,NaN,NaN,NaN,NaN,NaN
7,99774,66.01942,-149.07690,Stevens Village,AK,Alaska,True,NaN,11.0,0.4,...,America/Anchorage,112.536579,69.898496,Prospect Creek / Coldfoot,99774,NaN,NaN,NaN,NaN,NaN
8,86033,36.67265,-110.22883,Kayenta,AZ,Arizona,True,NaN,7407.0,2.4,...,America/Denver,112.521164,69.888922,Page Municipal Airport,86033,NaN,NaN,NaN,NaN,NaN
9,784,18.00544,-66.13392,Guayama,PR,Puerto Rico,True,NaN,40685.0,282.8,...,America/Puerto_Rico,109.181873,67.814828,Roosevelt Roads Puerto Rico,784,NaN,NaN,NaN,NaN,NaN


In [ ]:

file_type = 'AMY'
lat = 41.766595
lon = -88.318735
year = 2023
name= 'TEST_2023'
output_name = name + '.epw'

# Run your existing code with these parameters
data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
data_meteostat_merra2.to_csv(output_name, header=False, index=False)
with open(output_name, 'r') as original_file:
    data_content = original_file.read()
header_lines = create_header(data_meteostat_merra2, year, info_dict)
with open(output_name, 'w') as new_file:
    new_file.write("\n".join(header_lines) + "\n" + data_content)


In [ ]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QLineEdit, QPushButton, QVBoxLayout, QGridLayout, QMessageBox
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io

# Assuming all the previous functions are defined above or imported from another module


def run_individual_location(output_name, lat, lon, year, file_type, name):
    try:
        # Run your existing code with these parameters
        data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
        data_meteostat_merra2.to_csv(output_name, header=False, index=False)
        with open(output_name, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_name, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
        QMessageBox.information(window, "Success", f"Data saved successfully to {output_name}")
    except Exception as e:
        QMessageBox.critical(window, "Error", f"An error occurred: {str(e)}")


def on_run_clicked():
    lat = float(lat_input.text())
    lon = float(lon_input.text())
    year = int(year_input.text())
    file_type = file_type_input.text()
    name = name_input.text()
    output_name = output_name_input.text()
    
    run_individual_location(output_name, lat, lon, year, file_type, name)


# Initialize the application
app = QApplication(sys.argv)

# Create the main window
window = QWidget()
window.setWindowTitle("NOAA MERRA2 Data Processor")
window.setGeometry(100, 100, 400, 300)

# Create a grid layout
layout = QGridLayout()

# Add widgets for input fields
layout.addWidget(QLabel("Latitude:"), 0, 0)
lat_input = QLineEdit()
layout.addWidget(lat_input, 0, 1)
lat_input.setText("41.766595")

layout.addWidget(QLabel("Longitude:"), 1, 0)
lon_input = QLineEdit()
layout.addWidget(lon_input, 1, 1)
lon_input.setText("-88.318735")

layout.addWidget(QLabel("Year:"), 2, 0)
year_input = QLineEdit()
layout.addWidget(year_input, 2, 1)
year_input.setText("2023")

layout.addWidget(QLabel("File Type:"), 3, 0)
file_type_input = QLineEdit()
layout.addWidget(file_type_input, 3, 1)
file_type_input.setText("AMY")

layout.addWidget(QLabel("Name:"), 4, 0)
name_input = QLineEdit()
layout.addWidget(name_input, 4, 1)
name_input.setText("TEST_2023")

layout.addWidget(QLabel("Output File Name:"), 5, 0)
output_name_input = QLineEdit()
layout.addWidget(output_name_input, 5, 1)
output_name_input.setText("TEST_2023.epw")

# Add a run button
run_button = QPushButton("Run")
run_button.clicked.connect(on_run_clicked)
layout.addWidget(run_button, 6, 0, 1, 2)

# Set the layout for the main window
window.setLayout(layout)

# Show the window
window.show()

# Run the application's main loop
sys.exit(app.exec_())


## Figure out Zip Codes

In [ ]:
import pandas as pd
import numpy as np

# Define a function to calculate the Haversine distance between two points in km
def haversine(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula to calculate the distance
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

# 1) Open the file resources/zip_code_list.csv as a dataframe and call it zipcodes
zipcodes = pd.read_csv('resources/zip_codes_list.csv')

# 2) Open design_conditions.csv as a dataframe and call it dc
dc = pd.read_csv('resources/meteostat_stats.csv')

# Ensure latitude and longitude columns are in float format
zipcodes['lat'] = zipcodes['lat'].astype(float)
zipcodes['lng'] = zipcodes['lng'].astype(float)
dc['latitude'] = dc['latitude'].astype(float)
dc['longitude'] = dc['longitude'].astype(float)

# 3) Initialize columns in zipcodes dataframe for storing results
zipcodes['Distance'] = np.nan
zipcodes['Location'] = ""

# 4) Loop through all the rows in zipcodes
for idx, row in zipcodes.iterrows():
    lat1 = row['lat']
    lon1 = row['lng']

    # Calculate the distance to each location in the dc dataframe
    dc['Distance'] = dc.apply(lambda x: haversine(lat1, lon1, x['latitude'], x['longitude']), axis=1)

    # 5) Find the closest location in dc
    closest_location = dc.loc[dc['Distance'].idxmin()]

    # 6) Update the zipcodes dataframe with the closest location's details
    zipcodes.at[idx, 'Distance'] = closest_location['Distance']
    zipcodes.at[idx, 'Location'] = closest_location['name']

# Display the updated dataframe
zipcodes.to_csv('resources/updated_zip_code_list_again.csv', index=False)

In [ ]:
# Ensure the 'zip' column is a string and pad with zeros to make it 5 digits
zipcodes['zip0'] = zipcodes['zip'].astype(str).str.zfill(5)

zipcodes


In [ ]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

In [ ]:
import pandas as pd
import requests
from io import StringIO

# URL of the dataset containing ZIP codes and their respective coordinates
url = "https://raw.githubusercontent.com/scpike/us-state-county-zip/master/geo-data.csv"

# Fetching the CSV file from the URL
response = requests.get(url)
response.raise_for_status()  # Raises an error for bad responses

# Reading the CSV data into a pandas DataFrame
data = pd.read_csv(StringIO(response.text))


data.to_csv('resources/zip_codes_list.csv')


In [ ]:
import pandas as pd
import requests
from io import BytesIO
from zipfile import ZipFile

# URL of a dataset containing ZIP codes, latitude, and longitude
url = "https://simplemaps.com/static/data/us-zips/1.74/basic/simplemaps_uszips_basicv1.74.zip"

# Download the ZIP file with SSL verification disabled
response = requests.get(url, verify=False)  # Bypass SSL certificate verification
response.raise_for_status()  # Check if the request was successful

# Unzip the file and read the CSV
with ZipFile(BytesIO(response.content)) as zip_file:
    # Extract the CSV file within the ZIP
    with zip_file.open('uszips.csv') as file:
        zip_code_data = pd.read_csv(file)

# # Display the DataFrame to verify the contents
# print(zip_code_data.head())

# # Save the DataFrame to a local CSV file
zip_code_data.to_csv('resources/zip_codes_list.csv', index=False)
# print("Data saved to 'us_zip_codes_with_coordinates.csv'.")


In [ ]:
zip_code_data